# Notebook 52 — Ingesta de SQuAD v2 a la capa Bronze

En este laboratorio construimos un RAG pequeño pero completo para Databricks Free Edition.
El archivo `sampleqa100.json` conserva la estructura de SQuAD v2: título → contexto →
preguntas → respuestas.

Este primer notebook:

1. crea o reutiliza objetos gobernados por **Unity Catalog**;
2. lee el JSON desde un **Volume**;
3. aplana la jerarquía a una fila por pregunta;
4. descarta defensivamente preguntas imposibles;
5. selecciona únicamente la primera respuesta;
6. valida IDs, offsets y contenido antes de escribir una tabla Delta Bronze.


## 1. Parámetros gobernados

Los widgets permiten que cada estudiante use el catálogo y schema asignados por el docente.
Los valores predeterminados siguen la nomenclatura existente en este repositorio.

Si no tienes permiso para crear catálogos, selecciona en el widget `catalogo` uno que ya
exista y en el que tengas `USE CATALOG`, `USE SCHEMA`, `CREATE TABLE` y `CREATE VOLUME`.


In [ ]:
import re

dbutils.widgets.text("catalogo", "big_data_ii_2025", "1. Catálogo UC")
dbutils.widgets.text("esquema", "spark_examples", "2. Schema UC")
dbutils.widgets.text("volume", "agenteval_squadv2", "3. Volume UC")
dbutils.widgets.text("archivo", "sampleqa100.json", "4. Archivo JSON")

CATALOG = dbutils.widgets.get("catalogo").strip()
SCHEMA = dbutils.widgets.get("esquema").strip()
VOLUME = dbutils.widgets.get("volume").strip()
SOURCE_FILE = dbutils.widgets.get("archivo").strip()

for nombre, valor in {"catalogo": CATALOG, "esquema": SCHEMA, "volume": VOLUME}.items():
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", valor):
        raise ValueError(f"El valor de {nombre} no es un identificador simple válido: {valor!r}")

VOL = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
SOURCE_PATH = f"{VOL}/{SOURCE_FILE}"
T_BRONZE = f"{CATALOG}.{SCHEMA}.agenteval_squadv2_bronze"

print(f"Volume       : {VOL}")
print(f"Archivo      : {SOURCE_PATH}")
print(f"Tabla Bronze : {T_BRONZE}")


## 2. Crear catálogo, schema y Volume

Todos los datos persistentes de la práctica viven en Unity Catalog. La celda es idempotente:
puede ejecutarse más de una vez.


In [ ]:


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("Objetos de Unity Catalog listos.")


## 3. Verificar el archivo en el Volume

Si el archivo aún no está presente:

1. abre **Catalog** en la barra lateral;
2. navega a catálogo → schema → **Volumes** → `agenteval_squadv2`;
3. selecciona **Upload to this volume**;
4. sube `sampleqa100.json`;
5. vuelve a ejecutar esta celda.


In [ ]:
try:
    archivos = {f.name for f in dbutils.fs.ls(VOL)}
except Exception as e:
    raise RuntimeError(f"No se pudo listar el Volume {VOL}: {e}") from e

if SOURCE_FILE not in archivos:
    raise FileNotFoundError(
        f"No existe {SOURCE_PATH}. Sube '{SOURCE_FILE}' al Volume desde Catalog Explorer. "
        f"Archivos presentes: {sorted(archivos)}"
    )

print(f"Archivo encontrado: {SOURCE_PATH}")
print(f"Tamaño: {next(f.size for f in dbutils.fs.ls(VOL) if f.name == SOURCE_FILE):,} bytes")


## 4. Leer SQuAD v2 con schema explícito

Un schema explícito hace la lectura determinística y documenta la forma del dato. El
`multiLine=True` es obligatorio porque el JSON completo ocupa varias líneas.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

answer_schema = StructType([
    StructField("text", StringType(), False),
    StructField("answer_start", IntegerType(), False),
])

qa_schema = StructType([
    StructField("question", StringType(), False),
    StructField("id", StringType(), False),
    StructField("answers", ArrayType(answer_schema), True),
    StructField("is_impossible", BooleanType(), True),
])

paragraph_schema = StructType([
    StructField("context", StringType(), False),
    StructField("qas", ArrayType(qa_schema), False),
])

article_schema = StructType([
    StructField("title", StringType(), False),
    StructField("paragraphs", ArrayType(paragraph_schema), False),
])

squad_schema = StructType([
    StructField("version", StringType(), False),
    StructField("data", ArrayType(article_schema), False),
])

raw = (
    spark.read
    .option("multiLine", "true")
    .schema(squad_schema)
    .json(SOURCE_PATH)
)

raw.printSchema()
display(raw.select("version", F.size("data").alias("cantidad_titulos")))


## 5. Aplanar y seleccionar la primera respuesta

`posexplode` conserva las posiciones del título, contexto y pregunta. Estas posiciones son
útiles para trazabilidad del dato fuente. Aunque el archivo ya fue filtrado, volvemos a exigir
`is_impossible = false`: una capa Bronze no debe confiar ciegamente en el archivo recibido.


In [ ]:
articles = raw.select(
    F.col("version").alias("source_version"),
    F.posexplode("data").alias("article_pos", "article"),
)

paragraphs = articles.select(
    "source_version",
    "article_pos",
    F.col("article.title").alias("title"),
    F.posexplode("article.paragraphs").alias("paragraph_pos", "paragraph"),
)

questions = paragraphs.select(
    "source_version",
    "article_pos",
    "paragraph_pos",
    "title",
    F.col("paragraph.context").alias("context"),
    F.posexplode("paragraph.qas").alias("question_pos", "qa"),
)

bronze = (
    questions
    .filter((F.coalesce(F.col("qa.is_impossible"), F.lit(False)) == F.lit(False)))
    .filter(F.size("qa.answers") > 0)
    .withColumn("first_answer", F.element_at("qa.answers", 1))
    .select(
        F.col("qa.id").alias("question_id"),
        F.col("qa.question").alias("question"),
        "title",
        "context",
        F.col("first_answer.text").alias("answer_text"),
        F.col("first_answer.answer_start").alias("answer_start"),
        F.col("qa.is_impossible").alias("is_impossible"),
        "source_version",
        "article_pos",
        "paragraph_pos",
        "question_pos",
        F.lit(SOURCE_PATH).alias("source_path"),
        F.current_timestamp().alias("ingested_at"),
    )
)

print(f"Filas a escribir: {bronze.count():,}")
display(bronze.limit(10))


## 6. Pruebas antes de persistir

El offset de SQuAD es base cero. Spark `substring` es base uno; por eso sumamos 1. Si este
control falla, la respuesta ya no apunta al texto correcto dentro del contexto.


In [ ]:
n_rows = bronze.count()
n_ids = bronze.select("question_id").distinct().count()
n_nulls = bronze.filter(
    F.col("question_id").isNull()
    | F.col("question").isNull()
    | F.col("context").isNull()
    | F.col("answer_text").isNull()
    | F.col("answer_start").isNull()
).count()
n_bad_offsets = bronze.filter(
    F.expr(
        "substring(context, answer_start + 1, length(answer_text)) <> answer_text"
    )
).count()
n_impossible = bronze.filter(F.col("is_impossible") == F.lit(True)).count()

assert n_rows == 100, f"Se esperaban 100 preguntas y se encontraron {n_rows}."
assert n_ids == n_rows, f"Hay {n_rows - n_ids} question_id duplicados."
assert n_nulls == 0, f"Hay {n_nulls} filas con campos obligatorios nulos."
assert n_bad_offsets == 0, f"Hay {n_bad_offsets} respuestas cuyo offset no coincide."
assert n_impossible == 0, f"Quedaron {n_impossible} preguntas imposibles."

print("✓ 100 preguntas")
print("✓ IDs únicos")
print("✓ Sin nulos obligatorios")
print("✓ Sin preguntas imposibles")
print("✓ Los 100 offsets apuntan al texto de la respuesta")


## 7. Escribir la tabla Delta Bronze

Para que la práctica sea repetible, la ingesta usa `overwrite`. En producción se usaría una
clave de lote y `MERGE` para conservar el historial completo de ingestas.


In [ ]:
(
    bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(T_BRONZE)
)

spark.sql(
    f"""
    ALTER TABLE {T_BRONZE}
    SET TBLPROPERTIES (
      'layer' = 'bronze',
      'source.format' = 'squad_v2_json',
      'quality.answer_selection' = 'first_answer'
    )
    """
)

spark.sql(
    f"COMMENT ON TABLE {T_BRONZE} IS "
    "'Capa Bronze del laboratorio RAG: una fila por pregunta SQuAD v2.'"
)

print(f"Tabla creada: {T_BRONZE}")
display(spark.table(T_BRONZE).orderBy("article_pos", "paragraph_pos").limit(20))


## 8. Comprobación en la UI

En **Catalog Explorer**, abre la tabla `agenteval_squadv2_bronze` y revisa:

- **Overview**: tipo `MANAGED`, formato Delta y propietario;
- **Sample Data**: 100 filas;
- **Schema**: campos de trazabilidad y la primera respuesta;
- **History**: operación `WRITE`.

Siguiente notebook: **53 — Corpus Silver**.
